# Notebook 01 — Portal Ingestion & Standardisation

**Purpose:**  
Load raw tender data from four government procurement portals and transform each
into a common schema. Produces two output files per portal: `bid_data` and `financial_eval`.

**Inputs** (`data/raw/`):

| File | Portal |
|------|--------|
| `gem_bid_status_detail.xlsx` | GEM |
| `eprocure_result_tenders_detail.xlsx` | eProcure |
| `etenders_result_tenders_detail.xlsx` | eTenders India |
| `coal_india_tenders_details.xlsx` | Coal India |

**Outputs** (`data/processed/`):

| File | Description |
|------|-------------|
| `gem_bid_data.xlsx` | GEM bids in canonical schema |
| `gem_fe.xlsx` | GEM all-bidder rows (parsed from pipe-delimited cell) |
| `eprocure_bid_data.xlsx` | eProcure bids |
| `eprocure_financial_eval.xlsx` | eProcure bidder rows |
| `etender_bid_data.xlsx` | eTender bids |
| `etender_financial_eval.xlsx` | eTender bidder rows |
| `CoalIndia_bid_data.xlsx` | Coal India bids |
| `CoalIndia_financial_eval.xlsx` | Coal India bidder rows |

**Portal-specific notes:**
- **GEM**: All bidders encoded in a single pipe-delimited cell. Tenders converted to Reverse
  Auction (RA) may have a blank `Contract Seller` — backfilled from `L1 Bidder`.
- **eProcure / eTender**: Organisation strings use `||` as a hierarchy separator; the first
  token is extracted as `department_org`. State is inferred from embedded city names.
- **Coal India**: Ministry is always `'Ministry of Coal'`.

---

## 1. Imports & Setup

In [1]:
import sys
import re
import uuid
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.insert(0, str(Path('..').resolve()))

from config import RAW_FILES, PROCESSED_FILES
from src.cleaning import clean_company_name, clean_price, contract_to_days
from src.parsers import parse_date, parse_bidders
from src.extractors import extract_state_from_org
from src.schema import BID_COLUMNS

PROCESSED_FILES['gem_bids'].parent.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

Setup complete.


## 2. Utility — Shared Financial Eval Builder

eProcure, eTender, and Coal India all have identical `Rank / Seller / Price` columns.
One function builds the financial_eval table for all three.

In [2]:
def build_financial_eval(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a financial_eval table from portals that expose
    Rank, Seller, and Price columns directly.

    Parameters
    ----------
    df : Raw DataFrame from eProcure, eTender, or Coal India.

    Returns
    -------
    pd.DataFrame with columns: bid_number, rank_position, seller, price, status.
    """
    fe = df[['Bid Number', 'Rank', 'Seller', 'Price']].copy()
    fe = fe.rename(columns={
        'Bid Number': 'bid_number',
        'Rank'      : 'rank_position',
        'Seller'    : 'seller',
        'Price'     : 'price',
    })
    fe['status']        = None
    fe['seller']        = fe['seller'].astype(str).str.strip()
    fe['price']         = fe['price'].apply(clean_price)
    fe['rank_position'] = fe['rank_position'].astype(str).str.strip()
    return fe[['bid_number', 'rank_position', 'seller', 'price', 'status']]

## 3. GEM Portal

In [3]:
# ── Load ──────────────────────────────────────────────────────────────────────
gem_data = pd.read_excel(RAW_FILES['gem'])
print(f'GEM raw: {gem_data.shape}')

# ── Backfill RA bids where Contract Seller was not captured ───────────────────
# Tenders converted to Reverse Auction (RA) have a blank Contract Seller.
# The L1 Bidder column holds the equivalent winner in those cases.
gem_data.loc[gem_data['Contract Seller'].isna(), 'Contract Seller'] = gem_data['L1 Bidder']
gem_data.loc[gem_data['Contract Total (₹)'].isna(), 'Contract Total (₹)'] = gem_data['L1 Price (₹)']

n_blank = gem_data['Contract Seller'].isna().sum()
print(f'Contract Seller still blank after backfill: {n_blank}')
print('Note: residual blanks are RA bids where no winner was extractable.')

# ── Clean ─────────────────────────────────────────────────────────────────────
gem_data['L1 Bidder']       = gem_data['L1 Bidder'].apply(clean_company_name)
gem_data['Contract Seller'] = gem_data['Contract Seller'].apply(clean_company_name)
gem_data['Department']      = gem_data['Organisation Name'].fillna(gem_data['Department'])
gem_data['Contract Period'] = gem_data['Contract Period'].apply(contract_to_days)

# ── Build bid_data ─────────────────────────────────────────────────────────────
gem_bids = gem_data[[
    'Bid No', 'Title / Items', 'Ministry', 'Department',
    'Contract Period', 'Estimated Bid Value', 'Bid Start Date',
    'Bid End Date', 'Award Status', 'Contract Seller', 'Contract Total (₹)',
]].copy().rename(columns={
    'Bid No'             : 'bid_number',
    'Title / Items'      : 'title_description',
    'Ministry'           : 'ministry',
    'Department'         : 'department_org',
    'Contract Period'    : 'contract_period',
    'Estimated Bid Value': 'tender_value',
    'Bid Start Date'     : 'bid_start_date',
    'Bid End Date'       : 'bid_end_date',
    'Award Status'       : 'award_status',
    'Contract Seller'    : 'winner',
    'Contract Total (₹)' : 'winning_price',
})

gem_bids['tender_value']   = gem_bids['tender_value'].apply(clean_price)
gem_bids['winning_price']  = gem_bids['winning_price'].apply(clean_price)
gem_bids['bid_start_date'] = gem_bids['bid_start_date'].apply(parse_date)
gem_bids['bid_end_date']   = gem_bids['bid_end_date'].apply(parse_date)
gem_bids['state_name']     = np.nan

present_cols = [c for c in BID_COLUMNS if c in gem_bids.columns]
gem_bids     = gem_bids[present_cols].drop_duplicates(subset='bid_number').reset_index(drop=True)

# ── Parse financial_eval from pipe-delimited All Bidders column ────────────────
bidder_rows = []
for _, row in gem_data[['Bid No', 'All Bidders (Rank | Seller | Price | Status)']].iterrows():
    bidder_rows.extend(
        parse_bidders(row['Bid No'], row['All Bidders (Rank | Seller | Price | Status)'])
    )
gem_fe = pd.DataFrame(bidder_rows)

print(f'GEM bid_data: {gem_bids.shape}  |  GEM financial_eval: {gem_fe.shape}')

GEM raw: (3763, 19)
Contract Seller still blank after backfill: 50
Note: residual blanks are RA bids where no winner was extractable.
GEM bid_data: (3763, 12)  |  GEM financial_eval: (16199, 5)


## 4. eProcure Portal

In [4]:
eprocure_data = pd.read_excel(RAW_FILES['eprocure'])
print(f'eProcure raw: {eprocure_data.shape}')

eprocure_bids = eprocure_data[[
    'Bid Number', 'Work Description', 'Organization', 'Contract Period',
    'Tender Value', 'Published Date', 'Bid Submission End Date',
    'Tender Status', 'Awarded Bidder', 'Awarded Value',
]].copy().rename(columns={
    'Bid Number'             : 'bid_number',
    'Work Description'       : 'title_description',
    'Organization'           : 'department_org',
    'Contract Period'        : 'contract_period',
    'Tender Value'           : 'tender_value',
    'Published Date'         : 'bid_start_date',
    'Bid Submission End Date': 'bid_end_date',
    'Tender Status'          : 'award_status',
    'Awarded Bidder'         : 'winner',
    'Awarded Value'          : 'winning_price',
})

# Extract first level of org hierarchy
eprocure_bids['department_org'] = (
    eprocure_bids['department_org'].astype(str).str.split(r'\|\|').str[0].str.strip()
)

eprocure_bids['ministry']      = None
eprocure_bids['state_name']    = eprocure_data['Organization'].apply(extract_state_from_org)
eprocure_bids['tender_value']  = eprocure_bids['tender_value'].apply(clean_price)
eprocure_bids['winning_price'] = eprocure_bids['winning_price'].apply(clean_price)
eprocure_bids['contract_period'] = eprocure_bids['contract_period'].apply(contract_to_days)
eprocure_bids['bid_start_date']  = eprocure_bids['bid_start_date'].apply(parse_date)
eprocure_bids['bid_end_date']    = eprocure_bids['bid_end_date'].apply(parse_date)
eprocure_bids['winner']          = eprocure_bids['winner'].apply(clean_company_name)

present_cols  = [c for c in BID_COLUMNS if c in eprocure_bids.columns]
eprocure_bids = eprocure_bids[present_cols].drop_duplicates(subset='bid_number').reset_index(drop=True)
eprocure_fe   = build_financial_eval(eprocure_data)

print(f'eProcure bid_data: {eprocure_bids.shape}  |  financial_eval: {eprocure_fe.shape}')

eProcure raw: (1998, 16)
eProcure bid_data: (415, 12)  |  financial_eval: (1998, 5)


## 5. eTender Portal

In [5]:
etender_data = pd.read_excel(RAW_FILES['etender'])
print(f'eTender raw: {etender_data.shape}')

etender_bids = etender_data[[
    'Bid Number', 'Work Description', 'Organization', 'Contract Period',
    'Tender Value', 'Published Date', 'Bid Submission End Date',
    'Tender Status', 'Awarded Bidder', 'Awarded Value',
]].copy().rename(columns={
    'Bid Number'             : 'bid_number',
    'Work Description'       : 'title_description',
    'Organization'           : 'department_org',
    'Contract Period'        : 'contract_period',
    'Tender Value'           : 'tender_value',
    'Published Date'         : 'bid_start_date',
    'Bid Submission End Date': 'bid_end_date',
    'Tender Status'          : 'award_status',
    'Awarded Bidder'         : 'winner',
    'Awarded Value'          : 'winning_price',
})

etender_bids['department_org'] = (
    etender_bids['department_org'].astype(str).str.split(r'\|\|').str[0].str.strip()
)

etender_bids['ministry']        = None
etender_bids['state_name']      = etender_data['Organization'].apply(extract_state_from_org)
etender_bids['tender_value']    = etender_bids['tender_value'].apply(clean_price)
etender_bids['winning_price']   = etender_bids['winning_price'].apply(clean_price)
etender_bids['contract_period'] = etender_bids['contract_period'].apply(contract_to_days)
etender_bids['bid_start_date']  = etender_bids['bid_start_date'].apply(parse_date)
etender_bids['bid_end_date']    = etender_bids['bid_end_date'].apply(parse_date)
etender_bids['winner']          = etender_bids['winner'].apply(clean_company_name)

present_cols = [c for c in BID_COLUMNS if c in etender_bids.columns]
etender_bids = etender_bids[present_cols].drop_duplicates(subset='bid_number').reset_index(drop=True)
etender_fe   = build_financial_eval(etender_data)

print(f'eTender bid_data: {etender_bids.shape}  |  financial_eval: {etender_fe.shape}')

eTender raw: (1049, 16)
eTender bid_data: (292, 12)  |  financial_eval: (1049, 5)


## 6. Coal India Portal

In [6]:
coal_data = pd.read_excel(RAW_FILES['coal_india'])
print(f'Coal India raw: {coal_data.shape}')

coal_bids = coal_data[[
    'Bid Number', 'Work Description', 'Organization', 'Contract Period',
    'Estimated Bid Value', 'Bid Start Date', 'Bid End Date',
    'Tender Status', 'L1 Price (₹)', 'L1 Bidder',
]].copy().rename(columns={
    'Bid Number'         : 'bid_number',
    'Work Description'   : 'title_description',
    'Organization'       : 'department_org',
    'Contract Period'    : 'contract_period',
    'Estimated Bid Value': 'tender_value',
    'Bid Start Date'     : 'bid_start_date',
    'Bid End Date'       : 'bid_end_date',
    'Tender Status'      : 'award_status',
    'L1 Bidder'          : 'winner',
    'L1 Price (₹)'       : 'winning_price',
})

coal_bids['ministry']        = 'Ministry of Coal'
coal_bids['state_name']      = np.nan
coal_bids['tender_value']    = coal_bids['tender_value'].apply(clean_price)
coal_bids['winning_price']   = coal_bids['winning_price'].apply(clean_price)
coal_bids['contract_period'] = coal_bids['contract_period'].apply(contract_to_days)
coal_bids['bid_start_date']  = coal_bids['bid_start_date'].apply(parse_date)
coal_bids['bid_end_date']    = coal_bids['bid_end_date'].apply(parse_date)
coal_bids['winner']          = coal_bids['winner'].apply(clean_company_name)

present_cols = [c for c in BID_COLUMNS if c in coal_bids.columns]
coal_bids    = coal_bids[present_cols].drop_duplicates(subset='bid_number').reset_index(drop=True)
coal_fe      = build_financial_eval(coal_data)

print(f'Coal India bid_data: {coal_bids.shape}  |  financial_eval: {coal_fe.shape}')

Coal India raw: (3281, 16)
Coal India bid_data: (1220, 12)  |  financial_eval: (3281, 5)


## 7. Validation Summary

In [7]:
summary = pd.DataFrame([
    {'portal': 'GEM',        'bid_rows': len(gem_bids),      'fe_rows': len(gem_fe)},
    {'portal': 'eProcure',   'bid_rows': len(eprocure_bids), 'fe_rows': len(eprocure_fe)},
    {'portal': 'eTender',    'bid_rows': len(etender_bids),  'fe_rows': len(etender_fe)},
    {'portal': 'Coal India', 'bid_rows': len(coal_bids),     'fe_rows': len(coal_fe)},
])
print(summary.to_string(index=False))

assert all(summary['bid_rows'] > 0), "One or more portals produced zero bid rows."
assert all(summary['fe_rows']  > 0), "One or more portals produced zero financial eval rows."
print("\nAll validation checks passed.")

    portal  bid_rows  fe_rows
       GEM      3763    16199
  eProcure       415     1998
   eTender       292     1049
Coal India      1220     3281

All validation checks passed.


## 8. Export

In [8]:
exports = {
    PROCESSED_FILES['gem_bids']       : gem_bids,
    PROCESSED_FILES['gem_fe']         : gem_fe,
    PROCESSED_FILES['eprocure_bids']  : eprocure_bids,
    PROCESSED_FILES['eprocure_fe']    : eprocure_fe,
    PROCESSED_FILES['etender_bids']   : etender_bids,
    PROCESSED_FILES['etender_fe']     : etender_fe,
    PROCESSED_FILES['coal_india_bids']: coal_bids,
    PROCESSED_FILES['coal_india_fe']  : coal_fe,
}

for path, df in exports.items():
    df.to_excel(path, index=False)
    print(f"Exported {df.shape[0]:>5} rows  →  {path.name}")

print("\nNotebook 01 complete. Proceed to 02_data_merging.ipynb.")

Exported  3763 rows  →  gem_bid_data.xlsx
Exported 16199 rows  →  gem_fe.xlsx
Exported   415 rows  →  eprocure_bid_data.xlsx
Exported  1998 rows  →  eprocure_financial_eval.xlsx
Exported   292 rows  →  etender_bid_data.xlsx
Exported  1049 rows  →  etender_financial_eval.xlsx
Exported  1220 rows  →  CoalIndia_bid_data.xlsx
Exported  3281 rows  →  CoalIndia_financial_eval.xlsx

Notebook 01 complete. Proceed to 02_data_merging.ipynb.
